In [10]:
import modal
# from rich import print as rprint
import ipywidgets
from contextlib import contextmanager
import logging
from time import perf_counter

In [2]:
assert modal.__version__ > "0.49.0"
modal.__version__

'1.6.2'

In [11]:
@contextmanager
def timer(func_name: str):
    start = perf_counter()
    try:
        yield
    finally:
        print(f"{func_name} took {perf_counter() - start:.4f} seconds")

In [3]:
app = modal.App("example-get-started-v2")

In [7]:
@app.function()
def square(x):
    return x**2

@app.function()
def get_hostname_python_version():
    import socket
    import sys
    hostname = socket.gethostname()
    return f"Hostname: {hostname} - Python version: {sys.version}"

In [12]:
with app.run():
    with timer("square-local"):
        print(f"{ get_hostname_python_version.local()} - Run locally: {square.local(42)}") 
    
    with timer("square-remote"):
        print(f"{get_hostname_python_version.remote()} - Running remotely: {square.remote(42)}")

    for idx in range(5):
        with timer(f"square-remote-run#{idx}"):
            print(f"{get_hostname_python_version.remote()} - Running remotely: {square.remote(42)}")

Hostname: Mac.cable.rcn.com - Python version: 3.12.15 (main, Oct 21 2026, 17:38:27) [Clang 23.1.3 ] - Run locally: 1764
square-local took 0.0002 seconds
Hostname: modal - Python version: 3.12.6 (main, Sep 27 2024, 06:10:12) [GCC 12.2.0] - Running remotely: 1764
square-remote took 5.7957 seconds
Hostname: modal - Python version: 3.12.6 (main, Sep 27 2024, 06:10:12) [GCC 12.2.0] - Running remotely: 1764
square-remote-run#0 took 0.3503 seconds
Hostname: modal - Python version: 3.12.6 (main, Sep 27 2024, 06:10:12) [GCC 12.2.0] - Running remotely: 1764
square-remote-run#1 took 0.2598 seconds
Hostname: modal - Python version: 3.12.6 (main, Sep 27 2024, 06:10:12) [GCC 12.2.0] - Running remotely: 1764
square-remote-run#2 took 0.3295 seconds
Hostname: modal - Python version: 3.12.6 (main, Sep 27 2024, 06:10:12) [GCC 12.2.0] - Running remotely: 1764
square-remote-run#3 took 0.3109 seconds
Hostname: modal - Python version: 3.12.6 (main, Sep 27 2024, 06:10:12) [GCC 12.2.0] - Running remotely: 1764

In [13]:
## Modal function with a GPU attached
@app.function(gpu="any")
def hello_gpu():
    import subprocess

    
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        check=True,
    )
    return f"Hello from the remote GPU:\n{result.stdout.strip()}"

with app.run():
    result = hello_gpu.remote()
    print(result)

Hello from the remote GPU:
Sun Oct 25 16:44:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.57.04              KMD Version: 610.57.04     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:AC:00.0 Off |                    0 |
| N/A   34C    P8             32W /  350W |       6MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------